## LangChain 활용해보기

In [1]:
import os
import pathlib
import sys

# 노트북이 어느 위치에서 실행되든 backend/app 이 들어 있는 폴더(hanwha-agent)를 찾아 루트로 삼는다
# - 폴더 이름(day02)에 기대지 않으므로 다른 날짜 노트북에 복사해도 그대로 쓸 수 있다
here = pathlib.Path.cwd().resolve()
candidates = [here, *here.parents, here / "hanwha-agent"]
ROOT = next((p for p in candidates if (p / "backend" / "app").is_dir()), None)
if ROOT is None:
    raise RuntimeError(f"hanwha-agent 루트를 찾지 못했습니다. 현재 위치: {here}")

os.chdir(ROOT)                                  # 상대경로(.env 등)의 기준
SANDBOX = ROOT / "sandbox" / "w4" / "day02"

# app 패키지를 import 할 수 있게 backend 를 모듈 검색 경로 맨 앞에 넣는다
# - os.chdir 만으로는 import 경로가 바뀌지 않는다
BACKEND = str(ROOT / "backend")
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)

print("프로젝트 루트  :", ROOT)

프로젝트 루트  : C:\workspace\hanwha-agent


- 평범한 함수를 체인으로 잇기

In [2]:
# 체인에 끼울 함수 3개 — 아직 LangChain 과 아무 상관이 없는 평범한 함수다
# - 이 노트북은 모델을 부르지 않는다. call_llm 은 정해둔 문자열을 돌려주는 가짜다

def build_prompt(question: str) -> str:
    return f"[ 규정 질문 ]: {question}"

def call_llm(prompt: str) -> str:
    return '{"answer": "출장 일비는 1일 3만원입니다.", "doc_id": "DOC-HR-012"}'

def parser(text: str) -> dict:
    import json
    return json.loads(text)

In [3]:
from langchain_core.runnables import RunnableLambda

# RunnableLambda 로 감싸면 평범한 함수가 Runnable 이 된다
# - invoke / batch / stream 이 생기고, 파이프(|)로 이을 수 있게 된다
step = RunnableLambda(build_prompt)

print(type(step).__name__)
print(step.invoke("부산 출장 일비는?"))

RunnableLambda
[ 규정 질문 ]: 부산 출장 일비는?


In [4]:
# 파이프는 왼쪽 객체의 __or__ 가 처리한다
# - 평범한 함수에는 __or__ 가 없어서 함수끼리는 이어지지 않는다
try:
    build_prompt | call_llm
except TypeError as e:
    print("함수끼리 :", e)

# 맨 앞 하나만 Runnable 로 만들면 그 뒤는 LangChain 이 알아서 RunnableLambda 로 감싼다
# - call_llm 과 parser 는 감싸지 않았는데도 그대로 들어간다
pipeline = RunnableLambda(build_prompt) | call_llm | parser

print("파이프라인 :", type(pipeline).__name__)
print(pipeline.invoke("부산 출장 일비는?"))

함수끼리 : unsupported operand type(s) for |: 'function' and 'function'
파이프라인 : RunnableSequence
{'answer': '출장 일비는 1일 3만원입니다.', 'doc_id': 'DOC-HR-012'}


- batch : 한 건이 죽으면 전체가 죽는다

In [5]:
def lookup(payload: dict) -> str:
    # payload : {"doc_id": "..."} 조회할 문서 id 를 dict 로 받는다
    known = {"DOC-HR-012", "DOC-PU-007", "DOC-SE-002"}
    if payload["doc_id"] not in known:
        raise ValueError("모르는 문서...")
    return f"{payload['doc_id']} 조회 완료"

lookup_chain = RunnableLambda(lookup)

inputs = [
    {"doc_id": "DOC-HR-012"},
    {"doc_id": "DOC-XX-999"},   # 없는 doc_id — 여기서 터진다
    {"doc_id": "DOC-PU-007"},
    {"doc_id": "DOC-SE-002"},
]

# 기본값은 "하나라도 실패하면 전체 실패" 다
# - 멀쩡한 3건의 결과까지 못 받는다. 모델을 부르는 체인이었다면 그 3건은 이미 과금된 뒤다
try:
    lookup_chain.batch(inputs)
except ValueError as e:
    print("기본값 :", e, "→ 결과를 하나도 못 받는다")

기본값 : 모르는 문서... → 결과를 하나도 못 받는다


In [6]:
# return_exceptions=True 를 주면 실패한 자리에 예외 객체가 담긴 채로 전부 돌아온다
# - 순서가 그대로라 몇 번째 입력이 실패했는지 바로 안다
results = lookup_chain.batch(inputs, return_exceptions=True)

for one, sent in zip(results, inputs):
    mark = "실패" if isinstance(one, Exception) else "성공"
    print(f"  {mark}  {sent['doc_id']:<12} {one}")

  성공  DOC-HR-012   DOC-HR-012 조회 완료
  실패  DOC-XX-999   모르는 문서...
  성공  DOC-PU-007   DOC-PU-007 조회 완료
  성공  DOC-SE-002   DOC-SE-002 조회 완료


- 가짜 모델로 체인 확인하기

In [7]:
from langchain_core.language_models import FakeListChatModel
from langchain_core.output_parsers import StrOutputParser

# 돈을 쓰지 않고 체인 모양만 확인할 때 쓴다 (결과가 항상 같아 재현도 된다)
llm = FakeListChatModel(responses=["부산 출장 비용은 1일 2만원입니다."])

out = llm.invoke("부산 출장 일비는?")
print(type(out).__name__, ":", out.content)

# 파서를 붙이면 AIMessage 에서 content 만 꺼내 문자열로 준다
text_chain = llm | StrOutputParser()
print("파서 통과 :", text_chain.invoke("부산 출장 일비는?"))

AIMessage : 부산 출장 비용은 1일 2만원입니다.
파서 통과 : 부산 출장 비용은 1일 2만원입니다.


In [8]:
from langchain_core.language_models import GenericFakeChatModel
from langchain_core.messages import AIMessage

ANSWER = "일비 2만원 - 숙박 실비"

# stream 은 답을 조각으로 나눠 보낸다 — 다 만들어질 때까지 기다리지 않는다
# - messages 가 이터레이터라 한 번 흘리면 비어버린다. 아래에서 새로 만드는 이유다
gllm = GenericFakeChatModel(messages=iter([AIMessage(content=ANSWER)]))
pieces = list((gllm | StrOutputParser()).stream("부산 출장 일비는?"))

print("조각 개수 :", len(pieces))
print(pieces)

gllm2 = GenericFakeChatModel(messages=iter([AIMessage(content=ANSWER)]))
print("이어서 출력 :", end="")
for piece in (gllm2 | StrOutputParser()).stream("부산 출장 일비는?"):
    print(piece, end="", flush=True)   # end="" 라 줄내림 없이 옆으로 붙는다
print()

조각 개수 : 9
['일비', ' ', '2만원', ' ', '-', ' ', '숙박', ' ', '실비']
이어서 출력 :일비 2만원 - 숙박 실비


- RunnablePassthrough.assign : 받은 dict 에 키를 더한다

In [9]:
from langchain_core.runnables import RunnablePassthrough

# RunnablePassthrough() 와 .assign() 은 다르다
# - RunnablePassthrough()      : 받은 값을 그대로 흘린다
# - RunnablePassthrough.assign : 받은 dict 는 두고 키를 더해서 흘린다
enrich = RunnablePassthrough.assign(
    doc_id=lambda d: "DOC-HR-011",
    grade=lambda d: "일반",
)

print(type(enrich).__name__)
print(enrich.invoke({"question": "부산 출장 일비는?"}))

# question 이 살아 있으니 뒤에서 그대로 쓸 수 있다
# - RAG 에서 질문은 남겨두고 검색 결과만 붙일 때 이 모양이 된다
line = enrich | RunnableLambda(lambda d: f"[{d['doc_id']}/{d['grade']}] {d['question']}")
print(line.invoke({"question": "부산 출장 일비는?"}))

RunnableAssign
{'question': '부산 출장 일비는?', 'doc_id': 'DOC-HR-011', 'grade': '일반'}
[DOC-HR-011/일반] 부산 출장 일비는?


- 갈래로 나누기 : dict 는 자동으로 RunnableParallel 이 된다

In [10]:
from langchain_core.runnables import RunnableParallel

# 같은 입력을 여러 갈래로 보내고 결과를 dict 하나로 받는다
both = RunnableParallel(
    upper=RunnableLambda(lambda s: s.upper()),
    length=RunnableLambda(lambda s: len(s)),
)
print("RunnableParallel :", both.invoke("DOC-hr-002"))

# 파이프 안에서는 감쌀 필요가 없다 — dict 를 두면 LangChain 이 RunnableParallel 로 바꾼다
auto = RunnableLambda(lambda s: s) | {
    "upper": RunnableLambda(lambda s: s.upper()),
    "length": RunnableLambda(lambda s: len(s)),
}
print("dict 그대로      :", auto.invoke("DOC-hr-002"))
print("같은 결과인가    :", both.invoke("DOC-hr-002") == auto.invoke("DOC-hr-002"))

RunnableParallel : {'upper': 'DOC-HR-002', 'length': 10}
dict 그대로      : {'upper': 'DOC-HR-002', 'length': 10}
같은 결과인가    : True
